# Task

Design an in-process logging library that an application can call to record events. Each call should fan out to one or more configured destinations.

### Clarifying Questions

Q: What kind of operations does the library need to support?

The library needs to support two main categories of operations:

Logging calls — the application submits a log record at a given severity level (DEBUG, INFO, WARN, ERROR, FATAL), along with a message and optionally structured metadata/context (e.g. key-value pairs, timestamps).

Configuration — at startup, the application sets up one or more destinations, each with its own minimum severity threshold and its own format. Once configured, the library routes each incoming log record to whichever destinations have a threshold at or below the record's severity level.

Q: So, the number of destinations will be same as the number of severity levels? Say 5 destinations for (DEBUG, INFO, WARN, ERROR, FATAL)?

No, the number of destinations is independent of the number of severity levels. You can configure any number of destinations (one, five, ten, etc.), and each destination simply has its own minimum severity threshold that determines which records it accepts.

Q: If a destination has minimum severity of FATAL, only fatal events will be logged?

Yes, that's correct. A destination with a minimum severity of FATAL will only receive records logged at the FATAL level.

Q: What happens if events of other severity are passed to it?

They are simply filtered out — that destination ignores any record whose severity level is below its configured minimum threshold.

Q: What formats do the destination support?

At minimum, the library should support two built-in formats: plain text (a human-readable string) and JSON (structured key-value output). The design should be open to additional formats being added later without modifying existing code.

Q: Do the destinations need to store the events in some data store?

The destinations write records to an output sink — for example, a file on disk, the console/stdout, or a database. The library's job is to get the formatted bytes to that sink; whether the sink itself persists data long-term is a concern of the sink, not the logging library.

Q: Is the sink pass to the library during intialization?

Yes, the sink (e.g. a file handle, stdout, a database connection) is provided to the library at initialization time as part of configuring each destination.

Q: How are the destinations being referenced? Do they need a unique ID or a name?

For v1, destinations don't need a unique ID or name — they are simply a collection configured at startup and referenced internally by the library. There's no requirement for the application to look up or modify a specific destination by name after initialization.

Q: Does the library need to handle concurrent calls?

Yes, concurrent calls must be handled safely. Specifically, when multiple threads log simultaneously to the same destination, the bytes of one record must never interleave with the bytes of another record on that destination.

Q: How about duplicate events? Is there any chance that the caller will send duplicate events that the library needs to figure out?

Duplicate detection is out of scope for this library. The logging service simply accepts every call the application makes and routes it to the appropriate destinations — it has no responsibility for identifying or deduplicating repeated records.

Q: How many callers does the library need to support simultaneously?

The library doesn't impose a fixed limit on the number of concurrent callers — it should handle whatever concurrency the host application produces. The key constraint is simply that writes to any given destination must be safe under concurrent access, as mentioned earlier.

Q: Is there a size limit on each event or limit on the length of the message and metadata?

No size limit on the message or metadata — the library should accept whatever the application passes in.

Q: What are things that are out of scope?

Here's what's explicitly out of scope for v1:
- Hot-reloading config at runtime — destinations and their settings are fixed after initialization.
- Async or buffered writes — all writes are synchronous in v1, though the design should accommodate async later.
- Remote/network destinations — no shipping logs over the network in v1, but the design should be open to it.
- Hierarchical or named loggers — there's no logger hierarchy (e.g. no parent/child logger relationships) and no named logger lookup.
- Duplicate detection — the library does not identify or suppress repeated records.
- Size limits — no enforcement of message or metadata size.

Q: How should the library respond when the sink fails during a write?

For v1, treat a sink write failure as a best-effort operation — the library should not crash the application, but there is no requirement to retry, buffer, or recover the failed record. How failures are surfaced (silently swallowed, logged to stderr, etc.) is a design decision left to you.

Q: if an event is going to multiple destination and the sink fails in one of the destination, should the other destinations continue with the operation?

Yes, a failure on one destination should not prevent the library from attempting to write to the remaining destinations — each destination is treated independently.

# REQUIREMENTS

### My Answer - 1st attemp
1. The logging service logs events sent from applications to one or more destinations.
2. An event contains a severity level (DEBUG, INFO, WARN, ERROR, FATAL), and a message and optionally structured metadata/context (e.g. key-value pairs, timestamps).
3. The log event can be in either format of plain-text or JSON. The service will need to support additional formats in the future without any code modification.
4. Destinations are setup during initialization of the service. The service can support any number of destinations. 
5. Destination has configurations including a minimum severity threshold, format it supports and a sink where the logs will be output to.
6. An event will be written to one or more destinations as long as the event's severity is same or higher than the destination's minimum threshold.
7. The library needs to guarantee thread safety. When multiple threads log simultaneously to the same destination, the bytes of one record must never interleave with the bytes of another record on that destination.
8. There are no limits on the number of concurrent calls.


Out of Scope:
- Hot-reloading config at runtime — destinations and their settings are fixed after initialization.
- Async or buffered writes — all writes are synchronous in v1, though the design should accommodate async later.
- Remote/network destinations — no shipping logs over the network in v1, but the design should be open to it.
- Hierarchical or named loggers — there's no logger hierarchy (e.g. no parent/child logger relationships) and no named logger lookup.
- Duplicate detection — the library does not identify or suppress repeated records.
- Size limits — no enforcement of message or metadata size.

### Feedback
What went well

You captured the primary capabilities really well — logging calls with severity levels, fan-out to multiple destinations, per-destination thresholds, formats, and sinks are all clearly articulated. That's a solid foundation. 

Your out-of-scope section is excellent. You clearly internalized what you learned from the clarifying questions and translated it directly into boundaries, including async writes, network destinations, and hot-reload — all things that will keep your design focused. 
The thread safety requirement is spot-on. Calling out the non-interleaving guarantee specifically is exactly the right level of precision for a concurrency constraint. 

What needs improvement

One area that's missing is error handling at the destination level — specifically, what happens if a sink throws or fails during a write. Should the caller crash? Should other destinations still receive the record? This is worth capturing as a requirement because it directly affects how you'll structure the write loop in your design. 

You mentioned format is per-destination, but it's worth being explicit that format and destination type vary independently — any format can be paired with any sink. Right now requirement 5 reads as if a destination has 'a format it supports,' which could be interpreted as the sink dictating the format rather than the configuration choosing it freely. 

Minor suggestions

It'd be worth noting that each log record should capture a timestamp at the call site, not just rely on whatever the application passes in. This is a small but meaningful design decision that will come up when you define your record structure. 

### Sample Answer

Requirements:
1. Five severity levels: DEBUG < INFO < WARN < ERROR < FATAL
2. Each record carries timestamp, level, message, emitting thread name
3. Logger writes each record to one or more destinations, set at startup
4. Each destination has its own min-level threshold and its own format
   Format and destination type vary independently
5. Concurrent calls are safe. A record's bytes never interleave with
   another record's bytes on the same destination

Out of scope:
- Hot-reloading config at runtime
- Async / buffered writes
- Remote / network destinations in v1 (design should accommodate)
- Hierarchical / named loggers (com.app.service inheriting from com.app)


### My Answer - 2nd attempt

1. The logging service logs events sent from applications to one or more destinations.
2. An event contains a timestamp, name of the caller or application, a severity level (DEBUG, INFO, WARN, ERROR, FATAL), and a message and optionally structured metadata/context (e.g. key-value pairs, timestamps).
3. The log event can be in either format of plain-text or JSON. The service will need to support additional formats in the future without any code modification.
4. Destinations are setup during initialization of the service. The service can support any number of destinations. 
5. Destination has configurations including a minimum severity threshold, its own format and a sink where the logs will be output to. Format and destination type vary independently.
6. An event will be written to one or more destinations as long as the event's severity is same or higher than the destination's minimum threshold.
7. The library needs to guarantee thread safety. When multiple threads log simultaneously to the same destination, the bytes of one record must never interleave with the bytes of another record on that destination.
8. There are no limits on the number of concurrent calls.
9. If there is a sink write failure, the library and the calling application should not be affected. Also, the failure should not prevent the library from writing to other destinations.  


Out of Scope:
- Hot-reloading config at runtime — destinations and their settings are fixed after initialization.
- Async or buffered writes — all writes are synchronous in v1, though the design should accommodate async later.
- No retry, buffer, or recover the failed record during a sink write failure.
- Remote/network destinations — no shipping logs over the network in v1, but the design should be open to it.
- Hierarchical or named loggers — there's no logger hierarchy (e.g. no parent/child logger relationships) and no named logger lookup.
- Duplicate detection — the library does not identify or suppress repeated records.
- Size limits — no enforcement of message or metadata size.

### Feedback

What went well

You addressed the previous feedback on error handling really well — requirement 9 explicitly covers both that a sink failure shouldn't crash the caller and that it shouldn't block writes to other destinations. That's exactly the right level of precision. 
Your requirement 5 now explicitly states that format and destination type vary independently, which closes the ambiguity from last time about whether the sink dictates the format. That's a meaningful improvement. 
Your out-of-scope section is thorough and well-aligned with everything you learned from your clarifying questions — async writes, network destinations, hot-reload, and duplicate detection are all clearly bounded. 

Minor suggestions

One small thing worth noting: you mention the event includes a timestamp, but it would be worth clarifying that the timestamp is captured at the call site by the library itself, not just passed in by the caller. This is a subtle but important design decision that will come up when you define your log record structure. 

# Entities and Relationships

### My Answer - Entities 1st attempt
- Logger
- Event
- Destination

### Feedback
What went well

You correctly identified Logger, Event, and Destination as central pieces of the system — those are the right instincts and form a solid starting point. 
Recognizing Destination as its own entity shows you understand that each output target has its own configuration, which is a key part of this design. 
What needs improvement

Your list is missing a Formatter entity — this is the piece responsible for serializing a log event into plain-text or JSON, and it needs to be its own thing so you can swap formats independently of where logs go. 
You'd also want to pull out a Sink as its own entity — it's the interface that handles the actual byte-writing (console, file, etc.), and keeping it separate from Destination is what lets you add new output targets later without touching the rest of the system. 

Minor suggestions

Consider adding a LogLevel enum to represent the five severity levels — it gives you natural ordering for threshold comparisons and makes the severity concept explicit in code. 

### Sample Answer
- Logger
- LogRecord
- Destination
- Formatter (interface: PlainTextFormatter, JsonFormatter)
- Sink (interface: ConsoleSink, FileSink)
- LogLevel (enum: DEBUG, INFO, WARN, ERROR, FATAL)

### My Answer - Entities 2nd attempt
- Logger
- Event
- Destination
- Formatter (interface: PlainTextFormatter, JsonFormatter)
- Sink (interface: FileSink, ConsoleSink, DatabaseSink)
- LogLevel (enum: DEBUG, INFO, WARN, ERROR, FATAL)

### Feedback

What went well

You addressed the feedback from last time perfectly — you've added both Formatter (with PlainTextFormatter and JsonFormatter) and Sink (with FileSink, ConsoleSink, DatabaseSink) as explicit interfaces, which is exactly the right move for keeping format and output target independently swappable. 

Your LogLevel enum with all five severity levels (DEBUG, INFO, WARN, ERROR, FATAL) rounds out the list nicely and gives you a natural ordering for threshold comparisons. 
Logger, Event, and Destination are all present and correct, giving you a solid core to build the rest of the design around. 

### My Answer - Relationships
Logger class is the entry point of the library. During initialization, it takes in a list of destinations and keeps track of them. It exposes the log function which the applications will call to record events.  

Event class contains all the data related to the recorded event. 

Destination class contains the minimum severity threshold, a formatter, and a sink. 

### Feedback

What went well

You clearly identified Logger as the orchestrator — the entry point that holds the list of destinations and fans out log calls, which is exactly the right mental model. 
You nailed the composition inside Destination: it owns a Formatter and a Sink, which is the key insight that lets format and sink type vary independently without a class explosion. 
Your explanation that Logger creates an Event per log() call and passes it through to each Destination shows a clean understanding of the data flow across the whole system. 

### Sample Answer

"Logger is the orchestrator and the only class the application talks to. It owns an immutable list of Destinations, fixed at construction. On every log() call, Logger captures the timestamp and current thread's name, builds a LogRecord, and iterates over destinations handing each one the same record. Destination is where the per-target work lives - it composes a Formatter (for serialization) and a Sink (for actual byte-writing), holds its own minimum level threshold, and owns the per-destination lock. LogRecord is an immutable value object that flows through the system. Formatter and Sink are interfaces with concrete implementations (PlainText/JSON, Console/File), and any combination is just a constructor argument. There are no back-references and no shared mutable state across destinations - each one protects its own resource."


# Interface

class Logger:
    - _destinations
    + Logger(list<Destination>)
    + Log(log_level, message, metadata)

class Event:
    + Event(timestamp, caller_name, log_level, message, metadata)

class Destination:
    + Destination(min_level_threshold, formatter, sink)

abstract class Formatter:
    + Formatter()
    + Convert(Event) 

abstract class Sink:
    + Sink()
    + Write()